## sLTN Tutorial Series

This is the first of five tutorials introducing **sLTN** (Structural Logic Tensor Networks):

1. **Signature & Interpretation** (this notebook) — the vocabulary layer
2. [FOL Formulas](2-FOLFormulas.ipynb) — connectives and quantifiers
3. [Fuzzy Operators & Gradients](3-FuzzyOperators&Gradients.ipynb) — choosing operators for learning
4. [KB & Learning](4-KB&Learning.ipynb) — knowledge bases and training loops
5. [Structural LTN](5-StructuralLTN.ipynb) — reasoning over time, sequences, and graphs

Before running any of them, install the notebook dependencies:

```bash
pip install -e ".[notebooks]"
```

# Tutorial 1 — The Vocabulary: Sorts, Symbols, and Interpretation

In `sLTN`, a `Signature` declares the symbolic vocabulary of a logical language. These declarations specify the names, arities, and sorts of the symbols, but they do not contain tensors, neural networks, or other concrete computational objects.

Concrete meanings are assigned separately through an `Interpretation`. This separation between symbolic declaration and computational grounding is one of the core design principles of `sLTN`: it lets you write and reason about formulas independently of how they are eventually grounded, and swap in different groundings (e.g. a different neural architecture) without touching the logic itself.

| Symbol type | Declares | Typical grounding |
|---|---|---|
| `Sort` | A domain of individuals | A type object |
| `Constant` | A named fixed individual | A tensor or trainable parameter |
| `Variable` | A named batch of individuals | A tensor |
| `Predicate` | A truth-valued map into `[0, 1]` | A neural module or callable |
| `Function` | An individual-valued map | A neural module or callable |

An `Interpretation` also manages device placement. Calling `interp.to(device)` moves grounded tensors and modules to the selected device. New groundings added afterward are placed consistently on the same device (note that simple python callable containing tensors must be placed manually). 

In [ ]:
import torch
import torch.nn as nn
import sltn
from sltn import Signature, Interpretation

torch.manual_seed(0)
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print("device:", device)

## 1 — Signature and Sorts

A `Signature` defines the logical vocabulary used in a model.

The first symbols usually declared in a signature are **sorts**. A sort represents a domain of individuals, such as animals, people, images, embeddings, or real-valued vectors. Other symbols are declared with respect to sorts. For example, constants and variables belong to a sort, predicates take arguments of specified sorts, and functions map arguments from input sorts to an output sort. 

In the following cell, we create a signature, declare the sort `Animal`, create an interpretation, and ground `Animal` as a two-dimensional real-vector domain.

In [ ]:
# instantiating a signature
sig = Signature("tutorial")

# Declare a sort
sig.sort("Animal")

# instantiate an interpretation for the signature
interp = Interpretation(sig).to(device)

# Interpret sort on a type
R2 = sltn.Type("R2", shape=(2,)) # 2-D real vectors
interp["Animal"] = R2

## 2 — Constants

A constant is a **named fixed individual** of a given sort.

Since the sort `Animal` is interpreted here as a two-dimensional real-vector domain, constants of sort `Animal` are grounded as tensors whose final dimension is `2`. A constant can be grounded by an ordinary PyTorch tensor, or by an `nn.Parameter` when it should be trainable.

In [ ]:
# Declare constants of sort Animal
sig.constant("Fido", "Animal")
sig.constant("emb", "Animal")

# Interpret constants in ℝ²
interp["Fido"] = torch.tensor([2.1, 3.0])
interp["emb"] = nn.Parameter(torch.zeros(1, 2))

print(sig)
print(interp)

## 3 — Variables

A variable represents a **named batch of individuals**. Variables are used when evaluating formulas over collections of objects and when applying quantifiers.

Variables are declared in the signature and grounded in the interpretation. Once a variable has been grounded, formulas containing that variable can be evaluated directly: the interpretation resolves the grounding automatically.

A variable grounding can be replaced at any time by assigning a new tensor, e.g. to evaluate the same formula on another batch.

Internally, the interpretation converts grounded terms into `sltn.Tensor` objects. An `sltn.Tensor` wraps the underlying tensor together with named and typed axes, which allows `sLTN` to align variables, domains, and formula structure during evaluation.

Typical axis roles are:

- `variable`: batch axes associated with logical variables;
- `structural`: axes associated with declared structured dimensions, when present;
- `domain`: feature axes associated with the interpreted sort.

For example, `interp["x"] = torch.randn(10, 2)` for a variable `x` of sort `Animal` (grounded as 2-D vectors) produces a tensor shaped `[10, 2]`: one `variable` axis of size 10 (the batch of individuals) followed by one `domain` axis of size 2 (the feature dimensions of `Animal`). Evaluating a predicate over `x` keeps the `variable` axis and replaces the `domain` axis with a truth-value axis of size 1; quantifying over `x` then aggregates the `variable` axis away. Tutorial 5 introduces the third role, `structural`, for axes that are neither individuals nor features.

Reading a symbol from the interpretation returns the raw grounded value, such as a tensor or module. Calling the interpretation on a term returns the corresponding wrapped expression, which is useful for inspecting axis structure and checking that a symbol is interpreted correctly.

In [ ]:
sig.variable("x", "Animal")
sig.variable("y", "Animal")

# Bind tensor to the variable in the Interpretation
interp["x"] = torch.randn(10, 2)

In [ ]:
print("Interpretation: ", interp["x"]) # automatically placed on the same device as the Interpretation

In [ ]:
print("Evaluation of term 'x':", interp("x")) # This returns the underlying expression.

## 4 — Predicates

A predicate maps one or more individuals to a truth degree in `[0, 1]`. Predicates are declared in the signature by specifying their name and argument sorts. They are grounded in the interpretation using either a trainable neural module or a plain callable. A neural predicate is useful when the truth degree should be learned from data. A callable predicate is mostly useful for fixed, deterministic, or non-trainable definitions.
For a unary predicate, the grounding receives one tensor argument. For an `n`-ary predicate, it receives one tensor per argument. The output should contain truth degrees in `[0, 1]`, with a final truth-value dimension.


In [ ]:
# 4a — Neural-network predicate (trainable)
class IsA(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ELU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)

sig.predicate("P", ["Animal"])
interp["P"] = IsA()
print(interp["P"])

# 4b — Lambda predicate (no trainable params)
mu  = torch.tensor([0.5, 0.5], device=device)
P2  = sig.predicate("P2", ["Animal"])
interp["P2"] = lambda x: torch.exp(-torch.norm(x - mu, dim=-1, keepdim=True))

# 4c — Binary predicate
class Likes(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 8), nn.ELU(), nn.Linear(8, 1), nn.Sigmoid())
    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=-1))

Sim = sig.predicate("Sim", ["Animal", "Animal"])
interp["Sim"] = Likes()

## 5 — Functions

A function maps one or more individuals to an individual in a target sort. They return values in the domain of their declared output sort. Functions are declared in the signature by specifying their name, input sorts, and output sort. They are grounded in the interpretation using the same interface as predicates: either a neural module or a plain callable.
In this section, we define a neural function from `Animal` to an embedding sort, and a callable function that computes a vector-valued difference between two individuals of sort `Animal`.

In [ ]:
sig.sort("Emb")
interp["Emb"] = sltn.Type("R8", shape=(8,))

# Neural function: Animal → Emb
class EmbedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 8)
    def forward(self, x):
        return self.linear(x)

sig.function("embed", ["Animal"], "Emb")
interp["embed"] = EmbedNet()
print(interp["embed"])

# Lambda function: difference
sig.function("diff", ["Animal", "Animal"], "Animal")
interp["diff"] = lambda x, y: x - y

## 6 — Putting it together

An `Interpretation` connects the symbolic vocabulary declared in a `Signature` to concrete computational groundings.

The typical workflow is:

1. declare sorts, constants, variables, predicates, and functions in a signature;
2. assign concrete groundings in an interpretation;
3. parse a formula from a string;
4. evaluate the parsed formula with the interpretation.

Parsing resolves symbol names against the signature and produces a formula abstract syntax tree. Evaluating the formula grounds it into a truth-valued `sltn.Tensor`.

Every variable appearing in a formula must be grounded in the interpretation before evaluation. To evaluate on a different batch, rebind the variable with `interp["x"] = new_tensor`.


In [ ]:
# x and P2 are already bound in interp
phi    = sig.parse("P2(x)")
result = interp(phi)

print("truth values shape:", result.value.shape)     # [10, 1]
print("mean truth degree :", result.value.mean().item())

# Rebind x to evaluate on another batch — a plain tensor is wrapped automatically
interp["x"] = torch.randn(5, 2).to(device)
result_local = interp(phi)
print("local eval shape  :", result_local.value.shape)  # [5, 1]

# Binary predicate — ground y and evaluate
interp["y"] = torch.randn(7, 2)
phi_sim = sig.parse("Sim(x, y)")
result2 = interp(phi_sim)
print("Sim(x,y) shape    :", result2.value.shape)   # [10, 7, 1]


print(interp)

## 7 — Inspecting groundings

Groundings can be read back from the interpretation using the same indexing interface used to set them.

This is useful for inspecting trained modules, checking tensor shapes, and verifying device placement. After the interpretation has been moved to a device, grounded tensors and modules are kept on that device, including groundings added later.

In [ ]:
# ── Reading back groundings ───────────────────────────────────────────────────
# interp[sym] is symmetric with interp[sym] = value

model = interp["P"]                                   # retrieve bound nn.Module
print("P module         :", model)
print("P device         :", next(model.parameters()).device)

x_tensor = interp["x"]                               # retrieve bound tensor
print("x tensor shape   :", x_tensor.shape)
print("x device         :", x_tensor.device)

# All modules are already on the right device because of interp.to(device)
embed_model = interp["embed"]
print("embed device     :", next(embed_model.parameters()).device)

---

**Next:** [Tutorial 2 — FOL Formulas](2-FOLFormulas.ipynb) shows how to combine predicates into full formulas with connectives and quantifiers.